# Segmentação de pluma de fumaça com SAM (Segment Anything Model)

Atividade: avaliar a máscara que o SAM produz para uma pluma de fumaça, testar diferentes *prompts*,
melhorar a apresentação da máscara e justificar as melhorias com medições.

| Item do enunciado | Etapa |
|---|---|
| A máscara cobre a pluma? Inclui o que não é pluma? Deixa fumaça de fora? | 4 (métricas) e 5 (conclusão) |
| Caixa delimitadora + pontos positivos e negativos, **3 configurações** | 2 |
| Original, máscara binária e máscara sobreposta lado a lado | 3 |
| O que dificultou a segmentação (iluminação, contraste, fumaça × nuvem) | 1 e 4.2 |

**Como rodar:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU`, depois
`Executar tudo`. Sem GPU ele usa o modelo `vit_b`, mais leve.
Para usar sua própria imagem: `CFG["IMAGEM"] = "upload"` e ajuste os *prompts* na Etapa 2
(as coordenadas são frações de 0 a 1, e a Etapa 0 mostra a imagem com uma grade para você lê-las).

In [ ]:
# ============================================================
# ETAPA 0 — Instalação, imports, configuração e imagem
# ============================================================
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os, time, urllib.request
import numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

np.random.seed(42); torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white",
                     "axes.titlesize": 10, "axes.titleweight": "bold", "font.size": 9})

CFG = {
    "IMAGEM": "socal",     # "socal" | "yellowstone" | "upload" | caminho/URL
    "LADO_MAX": 1280,      # redimensiona o maior lado (velocidade e reprodutibilidade)
}

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
TIPO_MODELO = "vit_h" if DISPOSITIVO == "cuda" else "vit_b"
ARQ_CKPT = {"vit_h": "sam_vit_h_4b8939.pth", "vit_b": "sam_vit_b_01ec64.pth"}[TIPO_MODELO]
if not os.path.exists(ARQ_CKPT):
    print("baixando checkpoint...")
    urllib.request.urlretrieve(
        f"https://dl.fbaipublicfiles.com/segment_anything/{ARQ_CKPT}", ARQ_CKPT)
print(f"dispositivo={DISPOSITIVO} | modelo={TIPO_MODELO}")

os.makedirs("saida", exist_ok=True)

# ---- catálogo de imagens, com prompts calibrados em coordenadas FRACIONÁRIAS ----
# Usar fração (x/largura, y/altura) em vez de pixel é o que permite trocar de imagem
# ou de resolução sem que nenhum ponto escorregue para fora do objeto.
CATALOGO = {
    "socal": {  # céu azul + casas + haze no horizonte + placa ocluindo a pluma
        "url": ("https://commons.wikimedia.org/wiki/Special:FilePath/"
                "Smoke%20plume%20from%20SoCal%20wildfire%20October%202007.jpg?width=1280"),
        "caixa": (0.289, 0.266, 0.999, 0.625),
        "positivos": [(0.336, 0.479), (0.484, 0.438), (0.625, 0.385),
                      (0.742, 0.344), (0.320, 0.531)],
        "negativos": [(0.117, 0.156), (0.547, 0.156), (0.781, 0.125),   # céu
                      (0.234, 0.583), (0.117, 0.625), (0.570, 0.635),   # encosta, telhados
                      (0.469, 0.833), (0.883, 0.271)],                  # asfalto, placa
        "rois": {"pluma": (0.45, 0.33, 0.62, 0.45), "ceu": (0.05, 0.05, 0.30, 0.20),
                 "nuvem_haze": (0.60, 0.50, 0.78, 0.56), "edificacao": (0.02, 0.56, 0.22, 0.66)},
        "referencia": [(0.305, 0.552), (0.316, 0.490), (0.328, 0.448), (0.352, 0.425),
                       (0.391, 0.420), (0.426, 0.442), (0.438, 0.408), (0.469, 0.375),
                       (0.516, 0.359), (0.563, 0.359), (0.609, 0.344), (0.648, 0.313),
                       (0.688, 0.297), (0.727, 0.286), (0.773, 0.292), (0.820, 0.302),
                       (0.875, 0.302), (0.938, 0.297), (0.996, 0.302), (0.998, 0.392),
                       (0.930, 0.460), (0.855, 0.500), (0.781, 0.530), (0.703, 0.545),
                       (0.625, 0.540), (0.547, 0.526), (0.484, 0.516), (0.430, 0.521),
                       (0.359, 0.536)],
    },
    "yellowstone": {  # a fumaça gera pirocúmulos: caso extremo de "fumaça parece nuvem"
        "url": ("https://commons.wikimedia.org/wiki/Special:FilePath/Wildfire%20in%20Yellowstone"
                "%20National%20Park%20produces%20Pyrocumulus%20clouds1.jpg?width=1280"),
        "caixa": (0.641, 0.026, 0.999, 0.383),
        "positivos": [(0.750, 0.115), (0.805, 0.191), (0.898, 0.204),
                      (0.703, 0.255), (0.961, 0.293)],
        "negativos": [(0.234, 0.115), (0.391, 0.102),   # nuvens brancas de verdade
                      (0.047, 0.038), (0.961, 0.051),   # céu azul
                      (0.469, 0.510), (0.156, 0.319)],  # campo e fumaça distante difusa
        "rois": {"pluma": (0.72, 0.15, 0.86, 0.26), "ceu": (0.00, 0.00, 0.08, 0.06),
                 "nuvem_haze": (0.18, 0.07, 0.34, 0.16), "edificacao": (0.30, 0.50, 0.60, 0.60)},
        "referencia": None,
    },
}


def carregar_imagem(chave):
    if chave in CATALOGO:
        preset, destino = dict(CATALOGO[chave]), f"{chave}.jpg"
        if not os.path.exists(destino):
            req = urllib.request.Request(preset["url"], headers={"User-Agent": "colab-sam/1.0"})
            open(destino, "wb").write(urllib.request.urlopen(req).read())
        caminho = destino
    else:
        # imagem do aluno: prompts genéricos, que PRECISAM ser ajustados na Etapa 2
        preset = {"caixa": (0.25, 0.20, 0.95, 0.60),
                  "positivos": [(0.45, 0.35), (0.55, 0.40), (0.65, 0.35)],
                  "negativos": [(0.10, 0.10), (0.90, 0.10), (0.50, 0.90)],
                  "rois": {"pluma": (0.45, 0.32, 0.60, 0.44), "ceu": (0.05, 0.05, 0.25, 0.18),
                           "nuvem_haze": (0.62, 0.50, 0.86, 0.58),
                           "edificacao": (0.05, 0.78, 0.30, 0.92)},
                  "referencia": None}
        if chave == "upload":
            from google.colab import files
            caminho = list(files.upload().keys())[0]
        elif str(chave).startswith("http"):
            caminho = "imagem_usuario.jpg"
            req = urllib.request.Request(chave, headers={"User-Agent": "colab-sam/1.0"})
            open(caminho, "wb").write(urllib.request.urlopen(req).read())
        else:
            caminho = chave
        print("*** ATENÇÃO: prompts genéricos. Ajuste-os na Etapa 2 usando a grade abaixo. ***")

    rgb = cv2.cvtColor(cv2.imread(caminho), cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    esc = CFG["LADO_MAX"] / max(h, w)
    if esc < 1:
        rgb = cv2.resize(rgb, (round(w * esc), round(h * esc)), interpolation=cv2.INTER_AREA)
    return rgb, preset


IMG, PRESET = carregar_imagem(CFG["IMAGEM"])
ALT, LARG = IMG.shape[:2]


# ---- conversões fração -> pixel ----
def para_px(pts, w=None, h=None):
    p = np.asarray(pts, float).reshape(-1, 2)
    return np.stack([p[:, 0] * (w or LARG), p[:, 1] * (h or ALT)], axis=1)


def caixa_px(c):
    return np.array([c[0] * LARG, c[1] * ALT, c[2] * LARG, c[3] * ALT], float)


def roi_mascara(c):
    x0, y0, x1, y1 = caixa_px(c).astype(int)
    m = np.zeros((ALT, LARG), bool)
    m[max(y0, 0):y1, max(x0, 0):x1] = True
    return m


def poligono_mascara(v):
    m = np.zeros((ALT, LARG), np.uint8)
    cv2.fillPoly(m, [para_px(v).astype(np.int32)], 1)
    return m.astype(bool)


# ---- imagem com grade de coordenadas (para ler/ajustar prompts na mão) ----
fig, ax = plt.subplots(figsize=(11, 11 * ALT / LARG))
ax.imshow(IMG)
for x in range(0, LARG, 100):
    ax.axvline(x, color="yellow", lw=0.4); ax.text(x + 2, 14, f"{x}\n{x/LARG:.2f}", color="red", fontsize=6, va="top")
for y in range(0, ALT, 100):
    ax.axhline(y, color="yellow", lw=0.4); ax.text(2, y + 4, f"{y} / {y/ALT:.2f}", color="red", fontsize=6, va="top")
ax.set_title(f"{CFG['IMAGEM']} — {LARG}x{ALT}px (rótulos: pixel / fração)"); ax.axis("off")
fig.tight_layout(); fig.savefig("saida/00_grade.png", bbox_inches="tight"); plt.show()

---
# Etapa 1 — Por que esta imagem é difícil (medido, não opinado)

O enunciado pergunta *como a iluminação, o contraste e a semelhança entre fumaça e nuvens afetaram
o resultado*. Para responder com evidência, amostramos quatro regiões — **pluma**, **céu**,
**nuvem/haze** e **edificação** — e medimos:

* **contraste RMS** e **gradiente médio**: onde o gradiente é fraco, a borda prevista fica
  indefinida. Fumaça tem borda difusa por natureza física — é um meio semitransparente, não um
  objeto com superfície;
* **saturação**: fumaça e nuvem são ambas quase acromáticas, então cor não ajuda a separá-las;
* **distância de Bhattacharyya** entre os histogramas: 0 = regiões estatisticamente
  indistinguíveis. A comparação que importa é `pluma × nuvem_haze` contra `pluma × céu`.

In [ ]:
# ============================================================
# ETAPA 1 — Diagnóstico fotométrico
# ============================================================
HSV = cv2.cvtColor(IMG, cv2.COLOR_RGB2HSV)
CINZA = cv2.cvtColor(IMG, cv2.COLOR_RGB2GRAY)
GRAD = np.sqrt(cv2.Sobel(CINZA, cv2.CV_32F, 1, 0, 3) ** 2 +
               cv2.Sobel(CINZA, cv2.CV_32F, 0, 1, 3) ** 2)
ROIS = {n: roi_mascara(c) for n, c in PRESET["rois"].items()}

AMOSTRAS = {n: {"luminância": CINZA[m].astype(float),
                "saturação": HSV[:, :, 1][m].astype(float),
                "gradiente": GRAD[m]} for n, m in ROIS.items()}

TAB_REGIOES = pd.DataFrame([{
    "região": n,
    "luminância média": a["luminância"].mean(),
    "contraste RMS": a["luminância"].std(),
    "saturação média": a["saturação"].mean(),
    "gradiente médio": a["gradiente"].mean(),
} for n, a in AMOSTRAS.items()]).set_index("região").round(2)


def bhattacharyya(a, b, bins=64):
    """0 = distribuições idênticas; 8 = saturado (histogramas sem sobreposição)."""
    faixa = (float(min(a.min(), b.min())), float(max(a.max(), b.max())) + 1e-6)
    ha = np.histogram(a, bins, faixa)[0]; hb = np.histogram(b, bins, faixa)[0]
    ha = ha / (ha.sum() + 1e-12); hb = hb / (hb.sum() + 1e-12)
    return float(min(-np.log(np.sum(np.sqrt(ha * hb)) + 1e-12), 8.0))


TAB_SEP = pd.DataFrame({
    f"pluma × {o}": {f: bhattacharyya(AMOSTRAS["pluma"][f], AMOSTRAS[o][f])
                     for f in ("luminância", "saturação", "gradiente")}
    for o in AMOSTRAS if o != "pluma"}).T.round(3)

print("Estatísticas por região:"); display(TAB_REGIOES)
print("\nDistância de Bhattacharyya (0 = indistinguível):"); display(TAB_SEP)
d_nuvem, d_ceu = TAB_SEP.loc["pluma × nuvem_haze"].mean(), TAB_SEP.loc["pluma × ceu"].mean()
print(f"\n=> pluma×nuvem = {d_nuvem:.2f} e pluma×céu = {d_ceu:.2f}: "
      f"{'a nuvem/haze é o distrator mais perigoso.' if d_nuvem < d_ceu else 'o céu ainda compete.'}")

cores = {"pluma": "lime", "ceu": "deepskyblue", "nuvem_haze": "orange", "edificacao": "magenta"}
fig, axs = plt.subplots(1, 4, figsize=(19, 19 * ALT / LARG / 3.4))
axs[0].imshow(IMG)
for n, c in PRESET["rois"].items():
    x0, y0, x1, y1 = caixa_px(c)
    axs[0].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=cores[n], lw=2))
    axs[0].text(x0, y0 - 4, n, color=cores[n], fontsize=8, weight="bold")
axs[0].set_title("ROIs de amostragem")
axs[1].imshow(HSV[:, :, 1], cmap="viridis"); axs[1].set_title("Saturação\nfumaça e nuvem: ambas baixas")
axs[2].imshow(np.clip(GRAD, 0, 120), cmap="magma"); axs[2].set_title("Gradiente\nborda da pluma quase some")
for n, a in AMOSTRAS.items():
    axs[3].hist(a["luminância"], bins=50, alpha=0.5, density=True, label=n, color=cores[n])
axs[3].set_title("Histograma de luminância"); axs[3].legend(fontsize=7)
for a in axs[:3]:
    a.axis("off")
fig.tight_layout(); fig.savefig("saida/01_diagnostico.png", bbox_inches="tight"); plt.show()

---
# Etapa 2 — Três configurações de *prompt*

| | *Prompt* | Hipótese testada |
|---|---|---|
| **A** | só caixa delimitadora | o retângulo basta? Ele contém muito céu junto da pluma. |
| **B** | só 3 pontos positivos | sem limite espacial, até onde a região "espalha"? |
| **C** | caixa + 5 positivos + **negativos** | dizer o que **não** é pluma (céu, nuvem/haze, telhado, placa) resolve a ambiguidade? |

**Melhoria proposta e sua justificativa:** o SAM é agnóstico a classe — ele não tem o conceito
"fumaça", apenas devolve a região coerente que o *prompt* indica. Como a Etapa 1 mostra que pluma e
nuvem/haze têm estatística parecida, nenhuma caixa consegue excluir o distrator: ele está *dentro* da
caixa. O **ponto negativo** é o único elemento do *prompt* que empurra a fronteira de decisão para
fora de uma região específica — daí a expectativa de que C seja a melhor, o que a Etapa 4 confirma
ou refuta com números.

Também usamos `multimask_output=True` e ficamos com a máscara de maior score: para um *prompt*
ambíguo o SAM devolve três escalas plausíveis (parte / sub-objeto / objeto inteiro), e a ambiguidade
aqui é do problema, não um defeito do modelo.

In [ ]:
# ============================================================
# ETAPA 2 — Predição com os três prompts
# ============================================================
from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry[TIPO_MODELO](checkpoint=ARQ_CKPT).to(DISPOSITIVO)
predictor = SamPredictor(sam)
t0 = time.time(); predictor.set_image(IMG)
print(f"embedding da imagem em {time.time()-t0:.1f}s")

CAIXA = caixa_px(PRESET["caixa"])
POS = para_px(PRESET["positivos"])
NEG = para_px(PRESET["negativos"])
REFERENCIA = poligono_mascara(PRESET["referencia"]) if PRESET.get("referencia") else None


def prever(box=None, pontos=None, rotulos=None, pred=None):
    """Chama o SAM e devolve a máscara de maior score previsto."""
    pred = pred or predictor
    kw = {}
    if pontos is not None and len(pontos):
        kw["point_coords"] = np.asarray(pontos, float)
        kw["point_labels"] = np.asarray(rotulos, int)
    if box is not None:
        kw["box"] = np.asarray(box, float)[None, :]
    mascaras, scores, _ = pred.predict(multimask_output=True, **kw)
    i = int(np.argmax(scores))
    return mascaras[i].astype(bool), float(scores[i])


CONFIGS = {
    "A — só caixa": dict(box=CAIXA, pontos=None, rotulos=None),
    "B — só pontos (+)": dict(box=None, pontos=POS[:3], rotulos=np.ones(len(POS[:3]), int)),
    "C — caixa + (+) + (−)": dict(box=CAIXA, pontos=np.vstack([POS, NEG]),
                                  rotulos=np.concatenate([np.ones(len(POS), int),
                                                          np.zeros(len(NEG), int)])),
}

RESULTADOS = {}
for nome, cfg in CONFIGS.items():
    t0 = time.time()
    m, s = prever(**cfg)
    RESULTADOS[nome] = {"mascara": m, "score": s}
    print(f"{nome:24s} score={s:.3f}  área={100*m.mean():5.2f}%  ({time.time()-t0:.2f}s)")

---
# Etapa 3 — Apresentação da máscara

Três painéis por configuração: **original com os *prompts* desenhados**, **máscara binária** e
**máscara sobreposta**. As escolhas de apresentação e o motivo de cada uma:

* os *prompts* aparecem sobre a original — sem isso não dá para saber se o erro veio do modelo ou de
  um ponto mal colocado;
* a sobreposição usa **transparência + contorno**, porque preenchimento opaco esconde exatamente a
  borda, que é onde está o erro interessante;
* o painel binário traz o **percentual de área**, para dar escala ao que se vê.

In [ ]:
# ============================================================
# ETAPA 3 — Figuras lado a lado
# ============================================================

def desenhar_prompts(ax, cfg):
    if cfg.get("box") is not None:
        x0, y0, x1, y1 = cfg["box"]
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=2.2,
                               label="caixa"))
    if cfg.get("pontos") is not None and len(cfg["pontos"]):
        p, r = np.asarray(cfg["pontos"]), np.asarray(cfg["rotulos"])
        if (r == 1).any():
            ax.scatter(p[r == 1, 0], p[r == 1, 1], c="lime", marker="*", s=180, ec="k", lw=.7,
                       label="positivo (+)", zorder=5)
        if (r == 0).any():
            ax.scatter(p[r == 0, 0], p[r == 0, 1], c="red", marker="X", s=110, ec="k", lw=.7,
                       label="negativo (−)", zorder=5)


def sobrepor(img, mask, alpha=0.45):
    out = img.astype(np.float32).copy()
    out[mask] = (1 - alpha) * out[mask] + alpha * np.array([255, 60, 60], np.float32)
    out = out.astype(np.uint8)
    cont, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, cont, -1, (255, 230, 0), 2)
    return out


for i, (nome, cfg) in enumerate(CONFIGS.items(), 1):
    m, s = RESULTADOS[nome]["mascara"], RESULTADOS[nome]["score"]
    fig, axs = plt.subplots(1, 3, figsize=(16, 16 * ALT / LARG / 2.6))
    axs[0].imshow(IMG); desenhar_prompts(axs[0], cfg)
    axs[0].set_title(f"Original + prompts\n{nome}"); axs[0].legend(loc="lower right", fontsize=7)
    axs[1].imshow(m, cmap="gray"); axs[1].set_title(f"Máscara binária\nárea = {100*m.mean():.2f}%")
    axs[2].imshow(sobrepor(IMG, m))
    axs[2].set_title(f"Sobreposição (α=0.45 + contorno)\nscore previsto = {s:.3f}")
    for a in axs:
        a.axis("off")
    fig.tight_layout(); fig.savefig(f"saida/03_{i}.png", bbox_inches="tight"); plt.show()

# grade comparativa: as três configurações numa figura só
fig, axs = plt.subplots(3, 3, figsize=(13, 9.3))
for lin, (nome, cfg) in enumerate(CONFIGS.items()):
    m = RESULTADOS[nome]["mascara"]
    axs[lin, 0].imshow(IMG); desenhar_prompts(axs[lin, 0], cfg)
    axs[lin, 1].imshow(m, cmap="gray")
    axs[lin, 2].imshow(sobrepor(IMG, m))
    axs[lin, 0].set_ylabel(nome, fontsize=9)
    for a in axs[lin]:
        a.set_xticks([]); a.set_yticks([])
for c, t in enumerate(["original + prompts", "máscara binária", "sobreposição"]):
    axs[0, c].set_title(t)
fig.tight_layout(); fig.savefig("saida/03_comparativo.png", bbox_inches="tight"); plt.show()

---
# Etapa 4 — Avaliação no código

As três perguntas do enunciado viram três medidas:

| Pergunta | Métrica |
|---|---|
| A máscara **cobre** a pluma? | `recall` e `cobertura pluma (%)` |
| **Inclui** o que não é pluma? | `vazamento ceu / nuvem_haze / edificacao (%)` e `precisão` |
| **Deixa fumaça de fora?** | `1 − recall` e o mapa de erro (falso negativo em azul) |

> ⚠️ A máscara de referência da imagem `socal` é um polígono traçado à mão e **aproximado** — na
> base da pluma não existe fronteira objetiva entre fumaça e *haze*. Os IoU/Dice servem para comparar
> as configurações **entre si**, não como medida absoluta. Essa incerteza do próprio rótulo é parte
> da resposta à pergunta "por que é difícil". Em `yellowstone` não há referência, e as métricas sem
> referência continuam valendo.

A Etapa 4.2 fecha a pergunta sobre **iluminação e contraste**: a mesma cena, o mesmo *prompt*, com o
contraste reduzido — como o SAM recalcula o *embedding*, a máscara muda sem que nada na cena mude.

In [ ]:
# ============================================================
# ETAPA 4.1 — Tabela de métricas
# ============================================================

def iou(a, b):
    u = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / u) if u else 0.0


def metricas(m):
    d = {"área (%)": 100 * m.mean()}
    n, _, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    d["componentes"] = int(n - 1)
    cont, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    per = sum(cv2.arcLength(c, True) for c in cont)
    d["compacidade"] = float(4 * np.pi * m.sum() / per ** 2) if per else 0.0
    for nome, roi in ROIS.items():
        chave = "cobertura pluma (%)" if nome == "pluma" else f"vazamento {nome} (%)"
        d[chave] = 100 * np.logical_and(m, roi).sum() / roi.sum()
    if REFERENCIA is not None:
        vp = np.logical_and(m, REFERENCIA).sum()
        d["IoU"] = iou(m, REFERENCIA)
        d["Dice"] = float(2 * vp / (m.sum() + REFERENCIA.sum()))
        d["precisão"] = float(vp / m.sum()) if m.sum() else 0.0
        d["recall"] = float(vp / REFERENCIA.sum())
    return d


TABELA = pd.DataFrame({n: metricas(r["mascara"]) for n, r in RESULTADOS.items()}).T
TABELA.insert(0, "score SAM", [RESULTADOS[n]["score"] for n in TABELA.index])
TABELA = TABELA.round(3)
display(TABELA)
TABELA.to_csv("saida/04_metricas.csv")

# IoU entre as configurações: o quanto a resposta depende do prompt, e não da imagem
nomes = list(RESULTADOS)
for i, a in enumerate(nomes):
    for b in nomes[i + 1:]:
        print(f"IoU({a.split(' ')[0]}, {b.split(' ')[0]}) = "
              f"{iou(RESULTADOS[a]['mascara'], RESULTADOS[b]['mascara']):.3f}")

if REFERENCIA is not None:
    melhor = TABELA["IoU"].idxmax()
    mm = RESULTADOS[melhor]["mascara"]
    mapa = np.zeros((ALT, LARG, 3), np.uint8)
    mapa[np.logical_and(mm, REFERENCIA)] = (60, 200, 60)
    mapa[np.logical_and(mm, ~REFERENCIA)] = (230, 70, 70)
    mapa[np.logical_and(~mm, REFERENCIA)] = (70, 120, 240)
    fig, ax = plt.subplots(figsize=(9, 9 * ALT / LARG))
    ax.imshow((IMG * 0.45 + mapa * 0.55).astype(np.uint8)); ax.axis("off")
    ax.set_title(f"Mapa de erro — {melhor}\nverde = acerto | vermelho = incluiu o que não é pluma"
                 f" | azul = deixou fumaça de fora")
    fig.tight_layout(); fig.savefig("saida/04_erro.png", bbox_inches="tight"); plt.show()

In [ ]:
# ============================================================
# ETAPA 4.2 — Efeito do contraste (mesma cena, mesmo prompt)
# ============================================================
base = RESULTADOS["C — caixa + (+) + (−)"]["mascara"]
x = IMG.astype(np.float32) / 255
baixo_contraste = (np.clip((x - 0.5) * 0.5 + 0.5, 0, 1) * 255).astype(np.uint8)

p2 = SamPredictor(sam); p2.set_image(baixo_contraste)
m2, s2 = prever(**CONFIGS["C — caixa + (+) + (−)"], pred=p2)
print(f"contraste x0.5 -> IoU vs original = {iou(m2, base):.3f} | "
      f"área {100*(m2.mean()-base.mean()):+.2f} p.p. | score {s2:.3f} (era "
      f"{RESULTADOS['C — caixa + (+) + (−)']['score']:.3f})")

fig, axs = plt.subplots(1, 2, figsize=(13, 13 * ALT / LARG / 2))
axs[0].imshow(sobrepor(IMG, base)); axs[0].set_title("original (C)")
axs[1].imshow(sobrepor(baixo_contraste, m2))
axs[1].set_title(f"contraste reduzido à metade\nIoU vs original = {iou(m2, base):.2f}")
for a in axs:
    a.axis("off")
fig.tight_layout(); fig.savefig("saida/04_contraste.png", bbox_inches="tight"); plt.show()

---
# Etapa 5 — Conclusões

*(Confira os números da **sua** execução nas tabelas acima antes de entregar.)*

## O que o SAM acertou e o que errou

O SAM é **agnóstico a classe**: ele não tem o conceito "fumaça", apenas devolve a região coerente
indicada pelo *prompt*. Isso explica o comportamento das três configurações:

* **A (só caixa)** — a caixa que contém a pluma contém também muito céu e parte do horizonte. O
  modelo tende a devolver a região mais contrastada dentro do retângulo, então a máscara ou vaza para
  o céu ou encolhe para o núcleo denso da pluma.
* **B (só pontos positivos)** — sem limite espacial, a máscara cresce por continuidade de aparência e
  engole a faixa de *haze* do horizonte, que tem praticamente a mesma estatística da base da pluma
  (veja a distância de Bhattacharyya da Etapa 1).
* **C (caixa + positivos + negativos)** — os negativos em céu, nuvem/haze, telhado e placa são o
  único elemento capaz de excluir um distrator que está *dentro* da caixa. É por isso que as colunas
  `vazamento *` caem em C, e é a melhoria central deste trabalho.

O IoU entre as configurações (Etapa 4.1) é a medida mais direta disso: quanto menor ele for, mais a
resposta dependeu do *prompt* e menos da imagem.

## O que dificultou a segmentação

* **Semelhança fumaça × nuvem/haze** — fator dominante. Na Etapa 1 a distância de Bhattacharyya
  entre pluma e nuvem/haze fica bem abaixo da distância entre pluma e céu: as duas se sobrepõem no
  histograma, em luminância e em saturação. Na prática a fronteira é decidida pelo *prompt*, não
  pela imagem.
* **Contraste e borda difusa** — o mapa de gradiente mostra bordas fortes em telhados, poste e
  cordilheira, e borda fraca na pluma. Modelos de segmentação ancoram o contorno no gradiente; sem
  gradiente, a borda fica arbitrária e passa a depender da borda do retângulo que demos.
* **Iluminação** — a Etapa 4.2 mostra a mesma cena, com o mesmo *prompt*, dando outra máscara só
  porque o contraste mudou. Consequência prática: fotos do mesmo incêndio em horários diferentes não
  são comparáveis sem normalização.
* **Oclusão e contexto urbano** (`socal`) — a placa de trânsito corta a pluma em duas; sem um ponto
  negativo sobre ela, ela é absorvida pela máscara. Telhados e encosta competem na parte inferior da
  caixa.
* **Transparência** — perto da borda cada pixel é mistura de fumaça e fundo. Nenhuma máscara binária
  é "correta" ali: é limite do formato de saída, não do modelo.

## O que eu faria a seguir

* Repetir o protocolo em **N imagens** e reportar média ± desvio, em vez de um caso só.
* **Refinamento iterativo** (realimentar os *logits* em `mask_input`) e **pós-processamento
  morfológico** para estabilizar a borda e remover ilhas isoladas — com o cuidado de que manter só o
  maior componente conexo descartaria fumaça real quando a pluma está ocluída.
* Sair da máscara binária para uma **estimativa de transparência (alfa)**, que é o formato adequado
  para medir área de pluma em série temporal.
* Marcar uma **referência de consenso** (duas ou três pessoas traçando a mesma pluma): a concordância
  entre elas é o teto realista de IoU que qualquer modelo pode atingir aqui.

In [ ]:
# ============================================================
# ETAPA 6 — Exportar (figuras, máscaras e tabela)
# ============================================================
import shutil

for nome, r in RESULTADOS.items():
    rot = nome.split(" ")[0]
    cv2.imwrite(f"saida/mascara_{rot}.png", (r["mascara"] * 255).astype(np.uint8))
    cv2.imwrite(f"saida/sobreposicao_{rot}.png",
                cv2.cvtColor(sobrepor(IMG, r["mascara"]), cv2.COLOR_RGB2BGR))
shutil.make_archive("resultados_sam", "zip", "saida")
print("gerado:", sorted(os.listdir("saida")))

try:
    from google.colab import files
    files.download("resultados_sam.zip")
except Exception as e:
    print("(download automático indisponível fora do Colab)", e)

---
# Etapa 7 — Entrega no GitHub

**Caminho mais curto:** rode tudo primeiro, depois `Arquivo → Salvar uma cópia no GitHub`, escolha o
repositório e confirme. O link a entregar é o do `.ipynb` no GitHub
(`https://github.com/SEU_USUARIO/SEU_REPO/blob/main/Segmentacao_Pluma_SAM.ipynb`).

Rodar antes de salvar importa: o GitHub renderiza o `.ipynb` **com as saídas embutidas**, então o
professor vê as figuras e as tabelas sem precisar executar nada.

### Checklist

- [ ] Notebook executado do começo ao fim, com as saídas salvas.
- [ ] Etapa 3: original, binária e sobreposição para **cada** uma das três configurações.
- [ ] Etapa 4: tabela de métricas preenchida.
- [ ] Etapa 5: conclusões revisadas por você, de acordo com os números que saíram.
- [ ] Repositório público (ou com acesso para o professor).